# 10｜Process Anomaly Detection、Pure Cross-Lot OOD 與 Evidence Triangulation

前面幾本已經分別完成 SPC、multivariate structure、Virtual Metrology 和 model explainability。10 的工作不是再訓練一個新的預測模型，而是把不同 evidence 放到同一個 wafer / Lot 層級比較：**同一片 wafer 如果在某個方法裡很特別，在其他方法裡也會同樣特別嗎？**

這一版刻意把 evidence 分成兩大類，避免把不同資訊來源混在一起：

- **Process-only evidence**
  - Sequence-adjusted residual anomaly：扣掉 Lot 內 wafer sequence 後，residual pattern 是否少見。
  - Pure Cross-Lot OOD：整個 Lot 被當成 unseen Lot 時，process engineered features 相對其他 Lots 是否陌生。
  - SPC evidence：單一 process feature 是否出現 retrospective I-chart / MR-chart excursion。
- **Outcome / model-side evidence**
  - Quality variation：wafer 內 etch variation 是否偏高。
  - Nested OOF error：VM 對 held-out Lot wafer 的 prediction error。
  - VM-informed diagnostic：只用 09 找出的 model-sensitive sensors，看 wafer 是否在「模型特別依賴的方向」偏離。

最重要的修正是：**Pure Cross-Lot OOD 完全不使用 09 的 supervised sensor importance。** 09 的 top sensors 只放到獨立的 VM-informed diagnostic，不參與 OOD feature selection、threshold 或 candidate 判定。

最後也不把這些 evidence 硬加成一個 risk score。因為目前沒有 production calibration，把不同定義、不同尺度的訊號相加反而會造成過度解讀。


## 1. 先準備路徑、共用函式和安全檢查

這一步只是把後面會重複用到的工具準備好，包括：

- 自動找到專案根目錄。
- training-only near-constant filter。
- empirical percentile。
- Lot-cluster bootstrap Spearman CI。
- CSV boolean 欄位的安全轉換。

這裡還沒有做任何 anomaly / OOD 判定，所以看到 environment ready 只代表程式可以往下跑。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

from scipy.stats import spearmanr
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 200)

RTOL = 1e-7
ATOL = 1e-8
BOOTSTRAP_REPS = 2000
BOOTSTRAP_SEED = 20260926


def find_project_root(start=None):
    """從目前目錄往上找出包含 data/processed 的專案根目錄。"""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError(
        "找不到專案根目錄：從目前工作目錄往上都沒有 data/processed。"
        f"目前工作目錄：{start}"
    )


def relative_variation_mask(X, rtol=RTOL, atol=ATOL):
    """Cross-Lot OOD 每個 training fold 使用的 near-constant filter。"""
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or not np.isfinite(X).all():
        raise ValueError("X 必須是 finite 2D matrix")
    ref = X[0]
    tol = atol + rtol * np.abs(ref)
    return np.max(np.abs(X - ref), axis=0) > tol


def coerce_bool(series, name):
    """把 CSV 裡的 true/false、0/1 安全轉成真正 boolean。"""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    text = series.astype(str).str.strip().str.lower()
    mapped = text.map({"true": True, "false": False, "1": True, "0": False})
    bad = mapped.isna()
    if bad.any():
        bad_values = sorted(series[bad].astype(str).unique().tolist())
        raise ValueError(f"{name} 含無法辨識的布林值：{bad_values}")
    return mapped.astype(bool)


def empirical_percentile_vs_reference(values, reference):
    """計算每個分數相對 reference distribution 的 empirical percentile。"""
    values = np.asarray(values, dtype=float)
    reference = np.asarray(reference, dtype=float)
    reference = reference[np.isfinite(reference)]
    if len(reference) == 0:
        return np.full(len(values), np.nan)
    return np.asarray([(reference <= v).mean() for v in values], dtype=float)


def pca_reconstruction_rmse(Z, pca, n_components):
    """計算 standardized feature space 裡的 PCA reconstruction RMSE。"""
    scores = pca.transform(Z)[:, :n_components]
    recon = scores @ pca.components_[:n_components] + pca.mean_
    return np.sqrt(np.mean((Z - recon) ** 2, axis=1))


def cluster_bootstrap_spearman_ci(
    df,
    x_col,
    y_col,
    cluster_col="lot_number",
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    """計算描述性的 Spearman rho，並用 Lot-cluster bootstrap 估 95% interval。"""
    work = df[[cluster_col, x_col, y_col]].dropna().copy()
    if work[x_col].nunique() < 2 or work[y_col].nunique() < 2:
        return np.nan, np.nan, np.nan, 0

    x_all = work[x_col].to_numpy(dtype=float)
    y_all = work[y_col].to_numpy(dtype=float)
    g_all = work[cluster_col].to_numpy()
    rho = float(spearmanr(x_all, y_all).statistic)

    clusters = np.asarray(sorted(work[cluster_col].unique()))
    cluster_xy = {
        c: (x_all[g_all == c], y_all[g_all == c])
        for c in clusters
    }

    rng = np.random.default_rng(seed)
    boot_rhos = []
    for _ in range(n_boot):
        sampled_clusters = rng.choice(clusters, size=len(clusters), replace=True)
        x_boot = np.concatenate([cluster_xy[c][0] for c in sampled_clusters])
        y_boot = np.concatenate([cluster_xy[c][1] for c in sampled_clusters])
        if np.unique(x_boot).size < 2 or np.unique(y_boot).size < 2:
            continue
        value = spearmanr(x_boot, y_boot).statistic
        if np.isfinite(value):
            boot_rhos.append(float(value))

    if not boot_rhos:
        return rho, np.nan, np.nan, 0

    low, high = np.quantile(boot_rhos, [0.025, 0.975])
    return rho, float(low), float(high), len(boot_rhos)


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)
PROCESSED_DATA_DIR = (PROJECT_ROOT / "data" / "processed").resolve()
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("10 的分析環境已準備完成")
print("PROJECT_ROOT：", PROJECT_ROOT)
print("PROCESSED_DATA_DIR：", PROCESSED_DATA_DIR)


10 的分析環境已準備完成
PROJECT_ROOT： C:\Users\momo8\Desktop\semiconductor-etch-process-intelligence
PROCESSED_DATA_DIR： C:\Users\momo8\Desktop\semiconductor-etch-process-intelligence\data\processed


### 這個結果代表什麼？

這一格只是確認 Notebook 10 的執行環境。這次實際找到的專案根目錄是 `semiconductor-etch-process-intelligence`，而分析資料會從 `data/processed` 讀取。

所以目前只能確認「路徑和共用函式都準備好了」，**還沒有做任何 anomaly / OOD 判定，也沒有任何製程結論**。後面的結果都會直接使用這次專案裡真正的 processed artifacts。

## 2. 讀入 05～09 的正式 artifacts，先確認 wafer 沒有接錯

10 不重做前面的分析，而是直接使用上游正式 artifacts：

- 05：sequence-adjusted residual、SPC I/MR flags、duration flags
- 06：residual PCA variance，用來做 lineage check
- 07：extended process features 和 feature metadata
- 08：Nested OOF predictions
- 09：sensor-group permutation importance；**只供後面的 VM-informed diagnostic 使用，不參與 Pure OOD**

先檢查 Process、Quality、07 features、08 OOF 的 `experiment_key` 是否一一對得上，也確認 residual artifact 的 `valid_lot` 是真正 boolean。這一步如果接錯，後面的 overlap 分析都沒有意義。


In [2]:
paths = {
    "quality": PROCESSED_DATA_DIR / "wafer_quality_summary.csv",
    "process": PROCESSED_DATA_DIR / "process_wafer_summary.csv",
    "residual": PROCESSED_DATA_DIR / "spc_sequence_adjusted_residuals.csv",
    "spc_feature": PROCESSED_DATA_DIR / "spc_feature_summary.csv",
    "spc_wafer": PROCESSED_DATA_DIR / "spc_wafer_flag_summary.csv",
    "duration_i": PROCESSED_DATA_DIR / "spc_duration_flags.csv",
    "duration_mr": PROCESSED_DATA_DIR / "spc_duration_mr_flags.csv",
    "residual_pca_var_06": PROCESSED_DATA_DIR / "06_residual_pca_variance.csv",
    "process_extended": PROCESSED_DATA_DIR / "07_process_features_extended.csv",
    "feature_metadata": PROCESSED_DATA_DIR / "07_engineered_feature_metadata.csv",
    "oof": PROCESSED_DATA_DIR / "08_nested_oof_predictions.csv",
    "sensor_perm": PROCESSED_DATA_DIR / "09_sensor_grouped_permutation_importance.csv",
}

missing = [path for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "缺少 Notebook 10 必要 artifacts：" + ", ".join(str(path) for path in missing)
    )

quality = pd.read_csv(paths["quality"])
process = pd.read_csv(paths["process"])
spc_residuals = pd.read_csv(paths["residual"])
spc_feature_summary = pd.read_csv(paths["spc_feature"])
spc_wafer = pd.read_csv(paths["spc_wafer"])
duration_i = pd.read_csv(paths["duration_i"])
duration_mr = pd.read_csv(paths["duration_mr"])
residual_pca_var_06 = pd.read_csv(paths["residual_pca_var_06"])
process_extended = pd.read_csv(paths["process_extended"])
feature_metadata = pd.read_csv(paths["feature_metadata"])
nested_oof = pd.read_csv(paths["oof"])
sensor_perm = pd.read_csv(paths["sensor_perm"])
required_residual_cols = {
    "feature", "experiment_key", "lot_number", "wafer_number",
    "residual", "valid_lot",
}
missing_residual_cols = sorted(required_residual_cols.difference(spc_residuals.columns))
if missing_residual_cols:
    raise ValueError(
        "05 residual artifact 缺少必要欄位：" + ", ".join(missing_residual_cols)
    )

required_spc_summary_cols = {"feature", "valid_lots", "n_valid_wafers"}
missing_spc_cols = sorted(required_spc_summary_cols.difference(spc_feature_summary.columns))
if missing_spc_cols:
    raise ValueError(
        "05 spc_feature_summary 缺少 residual eligibility 欄位：" + ", ".join(missing_spc_cols)
    )

spc_residuals["valid_lot"] = coerce_bool(spc_residuals["valid_lot"], "valid_lot")

for name, df in {
    "quality": quality,
    "process": process,
    "process_extended": process_extended,
    "oof": nested_oof,
}.items():
    if "experiment_key" not in df.columns or df["experiment_key"].duplicated().any():
        raise ValueError(f"{name} experiment_key 缺失或重複")

if set(process["experiment_key"]) != set(process_extended["experiment_key"]):
    raise ValueError("03 process summary 與 07 process features 的 wafer identity 不一致")
if set(quality["experiment_key"]) != set(nested_oof["experiment_key"]):
    raise ValueError("02 quality 與 08 OOF 的 labeled-wafer identity 不一致")
if not set(quality["experiment_key"]).issubset(set(process["experiment_key"])):
    raise ValueError("Quality-labeled wafers 不是 Process wafers 的子集")

n_process = len(process)
n_quality = len(quality)
n_lots = int(process["lot_number"].nunique())

print("Process wafers：", n_process)
print("Quality-labeled / OOF wafers：", n_quality)
print("Lots：", n_lots)
print("05 residual rows：", len(spc_residuals))
print("09 sensor groups：", len(sensor_perm))


Process wafers： 96
Quality-labeled / OOF wafers： 88
Lots： 10
05 residual rows： 2592
09 sensor groups： 28


### 這個結果代表什麼？

這次正式讀入後，Process 母體共有 **96 片 wafers、10 個 Lots**，其中 **88 片**同時有 Quality label 和 Nested OOF prediction。05 的 sequence-adjusted residual artifact 有 **2,592 rows**，09 則有 **28 個 sensor groups**。

更重要的是，程式已經檢查：

- 03 的 Process wafers 和 07 engineered features 是同一批 96 片。
- 02 的 Quality wafers 和 08 Nested OOF predictions 是同一批 88 片。
- 88 片 labeled wafers 全部都包含在 96 片 Process wafers 裡。

因此後面把 residual、OOD、Quality 和 model error 合併時，不是靠 row order 硬接，而是用 `experiment_key` 對齊。這一步沒有發現 wafer identity 接錯的問題。

## 3. Sequence-Adjusted Residual Anomaly：只用 06 真正有效的 24 個 residual features

05 有些 feature 在某些 Lot 幾乎沒有變化，當時為了保留表格結構會留下 `residual = 0`，同時把 `valid_lot=False`。這種 0 只是 placeholder，不能拿來當真正的 residual variation。

所以這裡完全照 06 的規則，只留下「10 個 Lots 都 valid，而且 96 片 Process wafers 都有有效 residual」的 features。建立 residual matrix 後，我還會把 PCA explained variance 和 06 的 artifact 逐項比對；對不上就直接停，不繼續做 anomaly ranking。

Residual unusualness 用三個角度一起看：

1. Top-5 absolute standardized residual mean：有沒有少數 features 特別偏。
2. Ledoit-Wolf Mahalanobis distance：整體 multivariate position 是否偏離。
3. PCA reconstruction RMSE：這片 residual pattern 是否難用主要 joint structure 重建。

這裡的 95% threshold 是用同一批 96 片做 retrospective screening，所以只能拿來挑 investigation candidates，不能當 production alarm limit。

In [3]:
# Exact eligibility rule used by Notebook 06.
residual_features = (
    spc_feature_summary.loc[
        (spc_feature_summary["valid_lots"] == n_lots)
        & (spc_feature_summary["n_valid_wafers"] == n_process),
        "feature",
    ]
    .astype(str)
    .tolist()
)

excluded_residual_features = (
    spc_feature_summary.loc[
        ~spc_feature_summary["feature"].astype(str).isin(residual_features),
        ["feature", "valid_lots", "n_valid_wafers"],
    ]
    .sort_values(["valid_lots", "n_valid_wafers", "feature"])
    .reset_index(drop=True)
)

if not residual_features:
    raise ValueError("沒有任何 feature 符合 06 的 all-Lot-valid residual eligibility rule")

residual_long = spc_residuals.loc[
    spc_residuals["valid_lot"]
    & spc_residuals["feature"].astype(str).isin(residual_features)
].copy()

expected_rows = n_process * len(residual_features)
if len(residual_long) != expected_rows:
    raise ValueError(
        f"有效 residual rows = {len(residual_long)}，但預期 {expected_rows}；"
        "代表 feature × wafer coverage 與 06 定義不一致。"
    )

if residual_long.duplicated(["experiment_key", "feature"]).any():
    raise ValueError("有效 residual 資料存在重複 experiment_key × feature")

residual_matrix = (
    residual_long
    .pivot(index="experiment_key", columns="feature", values="residual")
    .reindex(process["experiment_key"])
)

if residual_matrix.isna().any().any():
    raise ValueError("Residual matrix 出現缺值；不可補 0 後繼續做 multivariate anomaly")

# Do not apply a second global near-constant filter here.
# The feature universe is intentionally fixed by the exact Notebook-06 eligibility rule.
Xr = residual_matrix.to_numpy(dtype=float)
scaler_r = StandardScaler().fit(Xr)
Zr = scaler_r.transform(Xr)

# 1) Top-5 absolute z mean
top_k = min(5, Zr.shape[1])
res_top5 = np.mean(np.sort(np.abs(Zr), axis=1)[:, -top_k:], axis=1)

# 2) Shrinkage Mahalanobis distance
lw_r = LedoitWolf().fit(Zr)
diff_r = Zr - lw_r.location_
res_mahal = np.sqrt(np.einsum("ij,jk,ik->i", diff_r, lw_r.precision_, diff_r))

# 3) PCA reconstruction RMSE using components required for >=90% variance
pca_r = PCA().fit(Zr)
cum_r = np.cumsum(pca_r.explained_variance_ratio_)
k_r = int(np.searchsorted(cum_r, 0.90) + 1)
k_r = min(k_r, pca_r.components_.shape[0])
res_recon = pca_reconstruction_rmse(Zr, pca_r, k_r)

# Strong lineage check against Notebook 06.
if len(residual_pca_var_06) != len(pca_r.explained_variance_ratio_):
    raise ValueError(
        "10 residual PCA feature count 與 06 不一致："
        f"10={len(pca_r.explained_variance_ratio_)}, 06={len(residual_pca_var_06)}"
    )

if not np.allclose(
    residual_pca_var_06["explained_variance_ratio"].to_numpy(dtype=float),
    pca_r.explained_variance_ratio_,
    rtol=1e-10,
    atol=1e-12,
):
    raise ValueError(
        "10 residual PCA explained variance 與 06 不一致。"
        "請檢查 residual feature universe / preprocessing，不能繼續產生 anomaly 結論。"
    )

k_r_06 = int(
    residual_pca_var_06.loc[
        residual_pca_var_06["cumulative_variance_ratio"] >= 0.90,
        "component",
    ].iloc[0]
)
if k_r != k_r_06:
    raise ValueError(f"10 / 06 的 90% residual PCA component count 不一致：{k_r} vs {k_r_06}")

# Retrospective screening thresholds: full 96-wafer empirical 95th percentile.
res_thresholds = {
    "residual_top5_abs_z_mean": float(np.quantile(res_top5, 0.95)),
    "residual_mahalanobis": float(np.quantile(res_mahal, 0.95)),
    "residual_pca_recon_rmse": float(np.quantile(res_recon, 0.95)),
}

res_driver_idx = np.argmax(np.abs(Zr), axis=1)
residual_feature_array = residual_matrix.columns.to_numpy(dtype=object)

residual_anomaly = process[["experiment_key", "lot_number", "wafer_number"]].copy()
residual_anomaly["residual_top5_abs_z_mean"] = res_top5
residual_anomaly["residual_mahalanobis"] = res_mahal
residual_anomaly["residual_pca_recon_rmse"] = res_recon
residual_anomaly["residual_driver_feature"] = [residual_feature_array[i] for i in res_driver_idx]
residual_anomaly["residual_driver_abs_z"] = np.abs(Zr[np.arange(len(Zr)), res_driver_idx])

for metric, threshold in res_thresholds.items():
    residual_anomaly[f"{metric}_threshold95"] = threshold
    residual_anomaly[f"{metric}_flag"] = residual_anomaly[metric] >= threshold
    residual_anomaly[f"{metric}_pct"] = residual_anomaly[metric].rank(
        method="average", pct=True
    )

residual_flag_cols = [
    "residual_top5_abs_z_mean_flag",
    "residual_mahalanobis_flag",
    "residual_pca_recon_rmse_flag",
]
residual_anomaly["residual_anomaly_consensus"] = (
    residual_anomaly[residual_flag_cols].sum(axis=1).astype(int)
)
residual_anomaly["residual_review_candidate"] = (
    residual_anomaly["residual_anomaly_consensus"] >= 2
)
residual_anomaly = residual_anomaly.sort_values(
    ["residual_anomaly_consensus", "residual_mahalanobis"],
    ascending=[False, False],
).reset_index(drop=True)

residual_candidate_ids = residual_anomaly.loc[
    residual_anomaly["residual_review_candidate"], "experiment_key"
].tolist()

top_res = residual_anomaly.iloc[0]

print("Residual features used：", len(residual_features))
print("Residual PCA components for >=90% variance：", k_r)
print("06 PCA lineage check：PASS")
print("Residual review candidates：", len(residual_candidate_ids))
print("Excluded residual features：")
display(excluded_residual_features)
display(residual_anomaly.head(12).round(5))


Residual features used： 24
Residual PCA components for >=90% variance： 12
06 PCA lineage check：PASS
Residual review candidates： 3
Excluded residual features：


,feature,valid_lots,n_valid_wafers
0,Stat3_Etch_MV_EpdIntensity,0,0
1,Stat3_Etch_MV_SourceRFTuningCapacitor,0,0
2,Stat3_Etch_MV_SourceRF2TuningCapacitor,6,60


,experiment_key,lot_number,wafer_number,residual_top5_abs_z_mean,residual_mahalanobis,residual_pca_recon_rmse,residual_driver_feature,residual_driver_abs_z,residual_top5_abs_z_mean_threshold95,residual_top5_abs_z_mean_flag,residual_top5_abs_z_mean_pct,residual_mahalanobis_threshold95,residual_mahalanobis_flag,residual_mahalanobis_pct,residual_pca_recon_rmse_threshold95,residual_pca_recon_rmse_flag,residual_pca_recon_rmse_pct,residual_anomaly_consensus,residual_review_candidate
0,2024-08-07_08,8,8,3.45133,6.87318,0.30226,Stat3_Etch_MV_Heater4Temp,4.95593,2.68472,True,1.00000,5.46331,True,1.00000,0.40752,False,0.65625,2,True
1,2024-08-07_09,8,9,2.80372,5.50179,0.32208,Stat3_Etch_MV_Heater2Temp,3.82231,2.68472,True,0.98958,5.46331,True,0.96875,0.40752,False,0.76042,2,True
2,2024-08-01_04,6,4,1.73424,5.49929,0.42877,Stat3_Etch_MV_Heater3Temp,2.17292,2.68472,False,0.76042,5.46331,True,0.95833,0.40752,True,0.96875,2,True
3,2024-08-07_10,8,10,2.42721,5.71178,0.23283,Stat3_Etch_MV_Heater4Temp,3.39809,2.68472,False,0.90625,5.46331,True,0.98958,0.40752,False,0.32292,1,False
4,2024-07-19_01,5,1,2.66922,5.64793,0.18257,Stat3_Etch_MV_HeliumBPFlow,2.81515,2.68472,False,0.93750,5.46331,True,0.97917,0.40752,False,0.08333,1,False
5,2024-07-05_01,2,1,2.69022,5.45132,0.33614,Stat3_Etch_MV_moriInnerCurrent,3.20405,2.68472,True,0.95833,5.46331,False,0.94792,0.40752,False,0.83333,1,False
6,2024-07-09_06,3,6,1.62975,5.01904,0.48362,Stat3_Etch_MV_HeliumBPPressure,2.19564,2.68472,False,0.65625,5.46331,False,0.87500,0.40752,True,1.00000,1,False
7,2024-07-09_02,3,2,1.63987,4.94869,0.45066,Stat3_Etch_MV_Gas7Flow,2.25188,2.68472,False,0.66667,5.46331,False,0.86458,0.40752,True,0.97917,1,False
8,2024-07-19_09,5,9,2.75672,4.85686,0.33496,Stat3_Etch_MV_Gas1Flow,3.37838,2.68472,True,0.97917,5.46331,False,0.83333,0.40752,False,0.81250,1,False
9,2024-07-09_05,3,5,2.73622,4.84375,0.26506,Stat3_Etch_MV_Gas5Flow,3.01189,2.68472,True,0.96875,5.46331,False,0.82292,0.40752,False,0.45833,1,False


### 這個結果代表什麼？

這一段最後真正使用 **24 個 sequence-adjusted residual features**。PCA 累積到 90% variance 需要 **12 個 components**，而且和 Notebook 06 的 PCA explained-variance 完全一致，所以 residual feature universe 和前面的分析有接上。

有 3 個 features 沒有進這次 multivariate residual screening：

- `Stat3_Etch_MV_EpdIntensity`：0 / 10 Lots 有有效 residual。
- `Stat3_Etch_MV_SourceRFTuningCapacitor`：0 / 10 Lots 有有效 residual。
- `Stat3_Etch_MV_SourceRF2TuningCapacitor`：只有 6 / 10 Lots、60 / 96 wafers 有效。

所以這裡沒有把 `valid_lot=False` 時留下的 placeholder 0 當成真正訊號。

三個 retrospective 95th-percentile thresholds 分別約為：

- Top-5 |z| mean：**2.68472**
- Mahalanobis distance：**5.46331**
- PCA reconstruction RMSE：**0.40752**

按照「三個指標至少 2 個超過 threshold」的規則，96 片中共有 **3 片 Residual Review Candidates**：

| Wafer | Lot | 通過的 residual 指標 | 主要 residual driver |
|---|---:|---|---|
| `2024-08-07_08` | 8 | Top-5 |z| + Mahalanobis | `Heater4Temp` |
| `2024-08-07_09` | 8 | Top-5 |z| + Mahalanobis | `Heater2Temp` |
| `2024-08-01_04` | 6 | Mahalanobis + PCA reconstruction | `Heater3Temp` |

這表示這 3 片在**扣掉各自 Lot 的線性 wafer-sequence trend 後**，剩下的 multivariate residual pattern 相對少見。其中 Lot 8 的 wafer 8、9 都被抓到，但兩片主要是 level / standardized-deviation 類指標突出；Lot 6 wafer 4 則是 Mahalanobis 和 PCA reconstruction 同時偏高，所以異常型態不完全一樣。

這裡只能把它們稱為 **review candidates**。95th percentile 是用目前 96 片資料做的 retrospective screening reference，不是 production control limit，也不能直接說這 3 片是壞品或找到真正 root cause。

## 4. Pure Leave-One-Lot-Out Cross-Lot OOD：固定 Outer Feature Universe，再做 Inner LOGO Calibration

這一段只回答一個問題：**如果某個 Lot 完全不參與 reference distribution 的建立，這個 Lot 的 process engineered features 對其他 Lots 來說有多陌生？**

Feature universe 只來自 07 的 process engineered features。**不使用 Mean Si Etch、不使用 08 prediction error，也不使用 09 top model-sensitive sensors。** 因此這一節才可以稱為 process-only Cross-Lot OOD。

這一版特別修正 calibration 的尺度一致性：

1. **Outer LOGO**：完整留出一個 test Lot。
2. 只用 outer-training Lots 做一次 near-constant filtering，得到這個 outer fold 的 **固定 feature universe**。
3. **Inner LOGO calibration**：在 outer-training Lots 裡再輪流留出一個 calibration Lot。所有 inner detectors 都使用步驟 2 的同一套欄位，不再各自重新換 feature mask。
4. 每個 inner detector 仍然只用自己的 inner-training Lots fit `StandardScaler`、Ledoit-Wolf covariance 和 PCA，再去 score 沒看過的 calibration Lot。
5. 把這些 cross-fitted reference scores 合併，建立 Top-5 |z|、Mahalanobis、PCA reconstruction RMSE 的 empirical 95% thresholds。
6. 最後用全部 outer-training Lots、同一套 fixed feature universe fit final detector，再 score 真正 held-out test Lot。

這樣做的重點是：**calibration scores 與 outer test scores 至少使用相同的 predictor definition**。PCA 保留幾個 components 仍由各 detector 自己的 training data 決定，因為那屬於 detector fitting，而不是 feature-universe selection。

Candidate 規則：三個 process-only detectors 至少 **2 / 3** 超過該 outer fold 的 cross-fitted 95% threshold。

限制仍要保留：Inner detector 通常用 8 Lots fit，而 final Outer detector用 9 Lots fit，因此這是 **group-aware cross-fitted empirical screening**，不是 exact conformal calibration、不是正式 production control limit，也不保證固定 false-positive rate。

In [4]:
meta_cols = ["experiment_key", "lot_number", "wafer_number"]
process_predictors = [c for c in process_extended.columns if c not in meta_cols]

Xp = process_extended[process_predictors].to_numpy(dtype=float)
gp = process_extended["lot_number"].to_numpy(dtype=int)

records = []
fold_threshold_records = []

for test_lot in sorted(np.unique(gp)):
    tr = np.where(gp != test_lot)[0]
    te = np.where(gp == test_lot)[0]

    Xtr_raw = Xp[tr]
    Xte_raw = Xp[te]
    gtr = gp[tr]

    # ------------------------------------------------------------------
    # Outer-training-only feature selection.
    # 這個 mask 一旦在 outer fold 決定，inner calibration 全程固定使用同一套欄位。
    # ------------------------------------------------------------------
    outer_mask = relative_variation_mask(Xtr_raw)
    kept = np.asarray(process_predictors, dtype=object)[outer_mask]
    Xtr_fixed = Xtr_raw[:, outer_mask]
    Xte_fixed = Xte_raw[:, outer_mask]

    if Xtr_fixed.shape[1] == 0:
        raise ValueError(f"Held-out Lot {test_lot} 後沒有任何可用 predictors")

    # ------------------------------------------------------------------
    # Inner LOGO calibration.
    # 每個 reference wafer 都由沒看過其 Lot 的 detector scoring，
    # 但 predictor universe 固定為 outer-training-only mask，不在 inner fold 重新選欄位。
    # ------------------------------------------------------------------
    tk_ref = np.full(len(tr), np.nan, dtype=float)
    md_ref = np.full(len(tr), np.nan, dtype=float)
    rr_ref = np.full(len(tr), np.nan, dtype=float)

    inner_pca_components = []
    inner_near_constant_counts = []

    reference_lots = sorted(np.unique(gtr))
    for calibration_lot in reference_lots:
        inner_fit = np.where(gtr != calibration_lot)[0]
        inner_val = np.where(gtr == calibration_lot)[0]

        Xi_fit = Xtr_fixed[inner_fit]
        Xi_val = Xtr_fixed[inner_val]

        if Xi_fit.shape[1] != Xtr_fixed.shape[1] or Xi_val.shape[1] != Xtr_fixed.shape[1]:
            raise ValueError(
                f"Outer test Lot {test_lot}, inner calibration Lot {calibration_lot} "
                "沒有維持固定 outer feature universe"
            )

        # Audit only：有些 outer-kept features 在某個 inner-training subset 內可能暫時近乎不變。
        # 這裡只記錄，不再刪除，避免 calibration 與 outer test 使用不同 predictor definition。
        inner_near_constant_counts.append(
            int((~relative_variation_mask(Xi_fit)).sum())
        )

        inner_scaler = StandardScaler().fit(Xi_fit)
        Zi_fit = inner_scaler.transform(Xi_fit)
        Zi_val = inner_scaler.transform(Xi_val)

        if not np.isfinite(Zi_fit).all() or not np.isfinite(Zi_val).all():
            raise ValueError(
                f"Outer test Lot {test_lot}, inner calibration Lot {calibration_lot} "
                "standardized matrix 含 NaN / Inf"
            )

        # Mahalanobis：covariance 只用 inner-training Lots fit。
        inner_lw = LedoitWolf().fit(Zi_fit)
        di_val = Zi_val - inner_lw.location_
        md_ref[inner_val] = np.sqrt(
            np.einsum("ij,jk,ik->i", di_val, inner_lw.precision_, di_val)
        )

        # PCA reconstruction RMSE：PCA 與 k 都只由 inner-training Lots 決定。
        inner_pca = PCA().fit(Zi_fit)
        inner_ratio = inner_pca.explained_variance_ratio_
        if not np.isfinite(inner_ratio).all():
            raise ValueError(
                f"Outer test Lot {test_lot}, inner calibration Lot {calibration_lot} "
                "PCA explained_variance_ratio_ 含 NaN / Inf"
            )
        inner_cum = np.cumsum(inner_ratio)
        kk_inner = int(np.searchsorted(inner_cum, 0.90) + 1)
        kk_inner = min(kk_inner, inner_pca.components_.shape[0])
        inner_pca_components.append(kk_inner)
        rr_ref[inner_val] = pca_reconstruction_rmse(
            Zi_val, inner_pca, kk_inner
        )

        # Top-5 absolute standardized feature deviation.
        # 因為 feature universe 固定，所以 top-k 的 feature dimension 與 outer detector 一致。
        top_k_inner = min(5, Zi_fit.shape[1])
        tk_ref[inner_val] = np.mean(
            np.sort(np.abs(Zi_val), axis=1)[:, -top_k_inner:],
            axis=1,
        )

    for name, values in {
        "Top-5 |z| cross-fitted reference": tk_ref,
        "Mahalanobis cross-fitted reference": md_ref,
        "PCA reconstruction cross-fitted reference": rr_ref,
    }.items():
        if not np.isfinite(values).all():
            raise ValueError(f"{name} 含 NaN / Inf")

    thresholds = {
        "ood_top5_abs_z_mean": float(np.quantile(tk_ref, 0.95)),
        "ood_mahalanobis": float(np.quantile(md_ref, 0.95)),
        "ood_pca_recon_rmse": float(np.quantile(rr_ref, 0.95)),
    }

    # ------------------------------------------------------------------
    # Final outer detector：同一套 fixed feature universe，
    # preprocessing / covariance / PCA 只用全部 outer-training Lots fit。
    # ------------------------------------------------------------------
    scaler = StandardScaler().fit(Xtr_fixed)
    Ztr = scaler.transform(Xtr_fixed)
    Zte = scaler.transform(Xte_fixed)

    if not np.isfinite(Ztr).all() or not np.isfinite(Zte).all():
        raise ValueError(f"Held-out Lot {test_lot} standardized matrix 含 NaN / Inf")

    lw = LedoitWolf().fit(Ztr)
    dte = Zte - lw.location_
    md_te = np.sqrt(
        np.einsum("ij,jk,ik->i", dte, lw.precision_, dte)
    )

    pca = PCA().fit(Ztr)
    ratio = pca.explained_variance_ratio_
    if not np.isfinite(ratio).all():
        raise ValueError(
            f"Held-out Lot {test_lot} PCA explained_variance_ratio_ 含 NaN / Inf"
        )
    cum = np.cumsum(ratio)
    kk = int(np.searchsorted(cum, 0.90) + 1)
    kk = min(kk, pca.components_.shape[0])
    rr_te = pca_reconstruction_rmse(Zte, pca, kk)

    top_k_fold = min(5, Ztr.shape[1])
    tk_te = np.mean(
        np.sort(np.abs(Zte), axis=1)[:, -top_k_fold:],
        axis=1,
    )

    fold_threshold_records.append({
        "test_lot": int(test_lot),
        "n_train_wafers": int(len(tr)),
        "n_test_wafers": int(len(te)),
        "n_calibration_wafers": int(len(tk_ref)),
        "n_calibration_lots": int(len(reference_lots)),
        "n_kept_predictors": int(len(kept)),
        "calibration_uses_fixed_outer_feature_universe": True,
        "inner_near_constant_predictors_min": int(min(inner_near_constant_counts)),
        "inner_near_constant_predictors_max": int(max(inner_near_constant_counts)),
        "pca_components_90pct": int(kk),
        "inner_pca_components_90pct_min": int(min(inner_pca_components)),
        "inner_pca_components_90pct_max": int(max(inner_pca_components)),
        "top5_threshold95_crossfit": thresholds["ood_top5_abs_z_mean"],
        "mahalanobis_threshold95_crossfit": thresholds["ood_mahalanobis"],
        "pca_recon_threshold95_crossfit": thresholds["ood_pca_recon_rmse"],
    })

    tk_pct = empirical_percentile_vs_reference(tk_te, tk_ref)
    md_pct = empirical_percentile_vs_reference(md_te, md_ref)
    rr_pct = empirical_percentile_vs_reference(rr_te, rr_ref)

    for local_i, global_i in enumerate(te):
        absz = np.abs(Zte[local_i])
        j = int(np.argmax(absz))
        driver = str(kept[j])

        top5_flag = bool(
            tk_te[local_i] >= thresholds["ood_top5_abs_z_mean"]
        )
        mahal_flag = bool(
            md_te[local_i] >= thresholds["ood_mahalanobis"]
        )
        recon_flag = bool(
            rr_te[local_i] >= thresholds["ood_pca_recon_rmse"]
        )
        consensus = int(top5_flag) + int(mahal_flag) + int(recon_flag)

        records.append({
            "experiment_key": process_extended.iloc[global_i]["experiment_key"],
            "lot_number": int(test_lot),
            "wafer_number": int(
                process_extended.iloc[global_i]["wafer_number"]
            ),
            "n_train_wafers": int(len(tr)),
            "n_kept_predictors": int(len(kept)),
            "pca_components_90pct": int(kk),
            "ood_top5_abs_z_mean": float(tk_te[local_i]),
            "ood_top5_abs_z_mean_threshold95_crossfit": thresholds[
                "ood_top5_abs_z_mean"
            ],
            "ood_top5_abs_z_mean_pct_vs_crossfit": float(tk_pct[local_i]),
            "ood_top5_abs_z_mean_flag": top5_flag,
            "ood_mahalanobis": float(md_te[local_i]),
            "ood_mahalanobis_threshold95_crossfit": thresholds[
                "ood_mahalanobis"
            ],
            "ood_mahalanobis_pct_vs_crossfit": float(md_pct[local_i]),
            "ood_mahalanobis_flag": mahal_flag,
            "ood_pca_recon_rmse": float(rr_te[local_i]),
            "ood_pca_recon_rmse_threshold95_crossfit": thresholds[
                "ood_pca_recon_rmse"
            ],
            "ood_pca_recon_rmse_pct_vs_crossfit": float(rr_pct[local_i]),
            "ood_pca_recon_rmse_flag": recon_flag,
            "cross_lot_ood_consensus": consensus,
            "cross_lot_ood_candidate": consensus >= 2,
            "ood_driver_feature": driver,
            "ood_driver_abs_z": float(absz[j]),
        })

cross_lot_ood = pd.DataFrame(records)
ood_fold_thresholds = (
    pd.DataFrame(fold_threshold_records)
    .sort_values("test_lot")
    .reset_index(drop=True)
)

if len(cross_lot_ood) != n_process:
    raise ValueError(
        f"Cross-Lot OOD rows={len(cross_lot_ood)}，應為 {n_process}"
    )
if cross_lot_ood["experiment_key"].duplicated().any():
    raise ValueError("Cross-Lot OOD 出現重複 experiment_key")
if set(cross_lot_ood["experiment_key"]) != set(process["experiment_key"]):
    raise ValueError(
        "Cross-Lot OOD wafer identity 與 Process population 不一致"
    )
if not ood_fold_thresholds[
    "calibration_uses_fixed_outer_feature_universe"
].all():
    raise ValueError("至少一個 outer fold 沒有固定 calibration feature universe")

cross_lot_ood = cross_lot_ood.sort_values(
    [
        "cross_lot_ood_consensus",
        "ood_mahalanobis_pct_vs_crossfit",
        "ood_mahalanobis",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

ood_candidate_ids = cross_lot_ood.loc[
    cross_lot_ood["cross_lot_ood_candidate"],
    "experiment_key",
].tolist()
ood_rate = len(ood_candidate_ids) / n_process

ood_candidates_by_lot = (
    cross_lot_ood.loc[cross_lot_ood["cross_lot_ood_candidate"]]
    .groupby("lot_number")
    .size()
    .rename("ood_candidate_count")
    .reset_index()
)

print("Pure OOD predictor source：07 process engineered features only")
print("Quality / 08 error / 09 importance used in Pure OOD：NO / NO / NO")
print("Calibration feature-universe consistency：PASS")
print(
    "Cross-Lot OOD candidates：",
    f"{len(ood_candidate_ids)} / {n_process} ({ood_rate:.1%})",
)
display(ood_fold_thresholds.round(5))
display(cross_lot_ood.head(12).round(5))

if len(ood_candidates_by_lot):
    print("OOD candidate distribution by Lot：")
    display(ood_candidates_by_lot)
else:
    print("這次沒有 wafer 符合 2 / 3 OOD consensus candidate 規則。")

Pure OOD predictor source：07 process engineered features only
Quality / 08 error / 09 importance used in Pure OOD：NO / NO / NO
Calibration feature-universe consistency：PASS
Cross-Lot OOD candidates： 5 / 96 (5.2%)


,test_lot,n_train_wafers,n_test_wafers,n_calibration_wafers,n_calibration_lots,n_kept_predictors,calibration_uses_fixed_outer_feature_universe,inner_near_constant_predictors_min,inner_near_constant_predictors_max,pca_components_90pct,inner_pca_components_90pct_min,inner_pca_components_90pct_max,top5_threshold95_crossfit,mahalanobis_threshold95_crossfit,pca_recon_threshold95_crossfit
0,1,86,10,86,9,98,True,0,1,24,23,23,3.88763,15.87066,0.83917
1,2,86,10,86,9,98,True,0,1,24,22,23,3.84986,17.09422,0.92548
2,3,86,10,86,9,98,True,0,1,24,22,23,4.02218,15.20590,0.78196
3,4,86,10,86,9,98,True,0,1,24,22,23,4.12995,15.42513,0.85796
4,5,86,10,86,9,98,True,0,1,23,22,23,4.02268,14.81044,0.78820
5,6,86,10,86,9,98,True,0,1,24,22,23,3.76975,15.20208,0.79894
6,7,90,6,90,9,98,True,0,1,24,22,23,3.80410,15.28354,0.79391
7,8,86,10,86,9,98,True,0,1,23,22,23,3.73843,13.31765,0.69605
8,9,86,10,86,9,98,True,0,1,24,22,23,3.81121,15.52298,0.83611
9,10,86,10,86,9,97,True,0,0,24,22,23,3.89705,15.58739,0.84938


,experiment_key,lot_number,wafer_number,n_train_wafers,n_kept_predictors,pca_components_90pct,ood_top5_abs_z_mean,ood_top5_abs_z_mean_threshold95_crossfit,ood_top5_abs_z_mean_pct_vs_crossfit,ood_top5_abs_z_mean_flag,ood_mahalanobis,ood_mahalanobis_threshold95_crossfit,ood_mahalanobis_pct_vs_crossfit,ood_mahalanobis_flag,ood_pca_recon_rmse,ood_pca_recon_rmse_threshold95_crossfit,ood_pca_recon_rmse_pct_vs_crossfit,ood_pca_recon_rmse_flag,cross_lot_ood_consensus,cross_lot_ood_candidate,ood_driver_feature,ood_driver_abs_z
0,2024-08-07_09,8,9,86,98,23,4.42031,3.73843,0.98837,True,22.65232,13.31765,1.00000,True,1.13973,0.69605,1.00000,True,3,True,late_minus_early__Heater2Temp,8.56836
1,2024-08-07_08,8,8,86,98,23,4.11588,3.73843,0.98837,True,19.69511,13.31765,1.00000,True,1.03189,0.69605,1.00000,True,3,True,late_minus_early__SourceRFPeakToPeak,5.56883
2,2024-07-09_01,3,1,86,98,24,5.56251,4.02218,1.00000,True,16.06515,15.20590,0.96512,True,0.84397,0.78196,0.96512,True,3,True,std__Gas7Flow,7.19899
3,2024-08-07_10,8,10,86,98,23,3.68014,3.73843,0.94186,False,16.50156,13.31765,0.98837,True,0.89373,0.69605,1.00000,True,2,True,late_minus_early__Heater2Temp,4.09647
4,2024-07-19_01,5,1,86,98,23,3.57404,4.02268,0.86047,False,15.64344,14.81044,0.95349,True,0.81928,0.78820,0.96512,True,2,True,std__PlatenRFTuningCapacitor,3.88996
5,2024-07-02_01,1,1,86,98,24,4.02094,3.88763,0.95349,True,14.92548,15.87066,0.93023,False,0.75581,0.83917,0.91860,False,1,False,late_minus_early__SourceRF2PeakToPeak,4.68875
6,2024-08-22_01,10,1,86,97,24,3.84914,3.89705,0.94186,False,13.56371,15.58739,0.93023,False,0.65474,0.84938,0.88372,False,0,False,std__Heater4Temp,5.18565
7,2024-07-05_01,2,1,86,98,24,3.83311,3.84986,0.94186,False,13.51032,17.09422,0.90698,False,0.67915,0.92548,0.86047,False,0,False,mean__moriInnerCurrent,4.39639
8,2024-08-05_01,7,1,90,98,24,2.91937,3.80410,0.75556,False,12.59091,15.28354,0.90000,False,0.67572,0.79391,0.92222,False,0,False,late_minus_early__Heater4Temp,3.22713
9,2024-08-21_09,9,9,86,98,24,3.47132,3.81121,0.88372,False,12.48465,15.52298,0.89535,False,0.60144,0.83611,0.80233,False,0,False,late_minus_early__PlatenRFTuningCapacitor,4.27195


OOD candidate distribution by Lot：


,lot_number,ood_candidate_count
0,3,1
1,5,1
2,8,3


### 這個結果代表什麼？

這次 Pure Cross-Lot OOD 已經完整通過 fixed-feature-universe 檢查，輸出顯示：

> `Calibration feature-universe consistency：PASS`

每個 outer fold 都先只用其他 Lots 決定一套 predictor universe，之後 inner LOGO calibration 和 final held-out-Lot scoring 都維持同一套欄位。大部分 outer folds 保留 **98 個 predictors**，只有 held-out Lot 10 時保留 **97 個**。Inner training subset 中暫時變成 near-constant 的欄位最多只有 1 個，但這些欄位只被記錄做 audit，沒有在 inner fold 重新刪除，所以 threshold 和 held-out score 的 feature definition 是一致的。

在這個修正版方法下，96 片 Process wafers 中共有 **5 片（5.2%）Cross-Lot OOD candidates**：

| Wafer | Lot | OOD consensus | 實際超過 threshold 的指標 | 主要 OOD driver |
|---|---:|---:|---|---|
| `2024-08-07_09` | 8 | **3 / 3** | Top-5 |z|、Mahalanobis、PCA reconstruction | `late_minus_early__Heater2Temp` |
| `2024-08-07_08` | 8 | **3 / 3** | Top-5 |z|、Mahalanobis、PCA reconstruction | `late_minus_early__SourceRFPeakToPeak` |
| `2024-07-09_01` | 3 | **3 / 3** | Top-5 |z|、Mahalanobis、PCA reconstruction | `std__Gas7Flow` |
| `2024-08-07_10` | 8 | **2 / 3** | Mahalanobis、PCA reconstruction | `late_minus_early__Heater2Temp` |
| `2024-07-19_01` | 5 | **2 / 3** | Mahalanobis、PCA reconstruction | `std__PlatenRFTuningCapacitor` |

候選分布是 **Lot 8 有 3 片、Lot 3 有 1 片、Lot 5 有 1 片**。其中 `2024-08-07_08` 和 `2024-08-07_09` 也同時是上一節的 residual-review candidates，但另外 3 片並沒有全部在 residual detector 被抓到。

這正好說明兩個方法看的東西不同：Residual anomaly 看的是「Lot 內扣掉 sequence trend 後還剩多少 unusual pattern」；Cross-Lot OOD 看的是「這片 wafer 相對其他 Lots 建立的 process distribution 有多陌生」。

這一節完全沒有使用 Quality target、08 prediction error 或 09 importance，所以這 5 片是 **process-only OOD screening result**。不過 OOD 仍然只是 distribution extrapolation，不等於 defect，也不能把表中的 driver feature 解讀成 causal root cause。

## 5. VM-Informed Diagnostic：09 的 model-sensitive sensors 只做模型脈絡診斷

Pure OOD 完成後，才另外引入 09 的 supervised explainability 結果。

這一節回答的是另一個問題：**wafer 是否在 VM 模型特別依賴的 sensor groups 上出現較大的 standardized deviation？**

因為 top sensors 是由 09 的 supervised VM models 與 Mean Si Etch target 關係推導出來，所以這個 channel 是 **target-informed / model-informed**。它不能被稱為純 unsupervised OOD，也不參與 `cross_lot_ood_candidate` 判定。

這裡雖然不是獨立 anomaly detector，但 `vm_informed_pct_vs_crossfit` 仍然需要有一致的比較基準。因此每個 outer fold 也採用和 Pure OOD 相同的原則：

1. 先由 outer-training Lots 決定固定的 VM-informed engineered-feature universe；
2. inner LOGO calibration 全程使用同一套 outer-selected features；
3. 每個 inner fold 只重新 fit 自己的 scaler，不重新換 feature mask；
4. held-out Lot 完全不參與 scaling 或 reference percentile 的建立。

這只能讓 **scaling / percentile comparison** 保持 held-out-Lot aware 與 feature-definition consistent；**不能消除 09 sensor selection 本身是 supervised 的事實**。

因此這裡只保留 continuous diagnostic score 與 percentile，不另外設定 anomaly candidate threshold。

In [5]:
meta_lookup = feature_metadata.set_index("engineered_feature")

# 09 的 global grouped permutation importance：明確標記為 supervised / target-informed。
top_model_sensors = (
    sensor_perm.sort_values("mean_delta_rmse", ascending=False)
    .head(5)["sensor"]
    .astype(str)
    .tolist()
)

sensor_for_predictor = {
    f: (
        str(meta_lookup.loc[f, "sensor_short"])
        if f in meta_lookup.index
        else "Unknown"
    )
    for f in process_predictors
}
selected_predictors = [
    f
    for f in process_predictors
    if sensor_for_predictor[f] in top_model_sensors
]

if not selected_predictors:
    raise ValueError(
        "09 top model-sensitive sensors 無法對到 07 engineered predictors"
    )

Xvm = process_extended[selected_predictors].to_numpy(dtype=float)
vm_records = []

for test_lot in sorted(np.unique(gp)):
    tr = np.where(gp != test_lot)[0]
    te = np.where(gp == test_lot)[0]
    gtr = gp[tr]

    Xtr_raw = Xvm[tr]
    Xte_raw = Xvm[te]

    # Outer-training-only feature selection.
    # 這個 feature universe 也固定給後面的 inner LOGO reference 使用。
    outer_mask = relative_variation_mask(Xtr_raw)
    outer_features = np.asarray(
        selected_predictors,
        dtype=object,
    )[outer_mask]
    Xtr_fixed = Xtr_raw[:, outer_mask]
    Xte_fixed = Xte_raw[:, outer_mask]

    if Xtr_fixed.shape[1] == 0:
        raise ValueError(
            f"VM-informed diagnostic: held-out Lot {test_lot} "
            "後沒有可用 predictors"
        )

    # Final outer scaling：held-out Lot 不參與。
    scaler = StandardScaler().fit(Xtr_fixed)
    Zte = scaler.transform(Xte_fixed)
    if not np.isfinite(Zte).all():
        raise ValueError(
            f"VM-informed diagnostic: held-out Lot {test_lot} "
            "standardized test matrix 含 NaN / Inf"
        )
    vm_score_te = np.max(np.abs(Zte), axis=1)

    # Inner LOGO cross-fitted reference distribution.
    # 注意：不再在 inner fold 重新做 feature mask。
    vm_ref = np.full(len(tr), np.nan, dtype=float)
    for calibration_lot in sorted(np.unique(gtr)):
        inner_fit = np.where(gtr != calibration_lot)[0]
        inner_val = np.where(gtr == calibration_lot)[0]

        Xi_fit = Xtr_fixed[inner_fit]
        Xi_val = Xtr_fixed[inner_val]

        if (
            Xi_fit.shape[1] != Xtr_fixed.shape[1]
            or Xi_val.shape[1] != Xtr_fixed.shape[1]
        ):
            raise ValueError(
                f"VM-informed diagnostic: outer Lot {test_lot}, "
                f"inner Lot {calibration_lot} 沒有維持固定 outer feature universe"
            )

        inner_scaler = StandardScaler().fit(Xi_fit)
        Zi_val = inner_scaler.transform(Xi_val)
        if not np.isfinite(Zi_val).all():
            raise ValueError(
                f"VM-informed diagnostic: outer Lot {test_lot}, "
                f"inner Lot {calibration_lot} standardized matrix 含 NaN / Inf"
            )
        vm_ref[inner_val] = np.max(np.abs(Zi_val), axis=1)

    if not np.isfinite(vm_ref).all():
        raise ValueError(
            "VM-informed cross-fitted reference 含 NaN / Inf"
        )

    vm_pct = empirical_percentile_vs_reference(
        vm_score_te,
        vm_ref,
    )

    for local_i, global_i in enumerate(te):
        absz = np.abs(Zte[local_i])
        j = int(np.argmax(absz))
        driver_feature = str(outer_features[j])

        vm_records.append({
            "experiment_key": process_extended.iloc[global_i][
                "experiment_key"
            ],
            "lot_number": int(test_lot),
            "wafer_number": int(
                process_extended.iloc[global_i]["wafer_number"]
            ),
            "vm_informed_n_predictors": int(len(outer_features)),
            "vm_informed_max_abs_z": float(vm_score_te[local_i]),
            "vm_informed_pct_vs_crossfit": float(vm_pct[local_i]),
            "vm_informed_driver_feature": driver_feature,
            "vm_informed_driver_sensor": sensor_for_predictor[
                driver_feature
            ],
        })

vm_informed = pd.DataFrame(vm_records)

if (
    len(vm_informed) != n_process
    or vm_informed["experiment_key"].duplicated().any()
):
    raise ValueError(
        "VM-informed diagnostic wafer coverage 不正確"
    )

vm_informed = vm_informed.sort_values(
    [
        "vm_informed_pct_vs_crossfit",
        "vm_informed_max_abs_z",
    ],
    ascending=[False, False],
).reset_index(drop=True)

print("09 top model-sensitive sensors：", top_model_sensors)
print("Mapped engineered predictors：", len(selected_predictors))
print("VM-informed inner calibration fixed outer feature universe：PASS")
print("這一節建立 candidate flag：NO")
display(vm_informed.head(12).round(5))

09 top model-sensitive sensors： ['PlatenRFTuningCapacitor', 'MainActiveWindow', 'ForeLinePressure', 'SourceRFReflectedPower', 'PlatenRFReflectedPower']
Mapped engineered predictors： 17
VM-informed inner calibration fixed outer feature universe：PASS
這一節建立 candidate flag：NO


,experiment_key,lot_number,wafer_number,vm_informed_n_predictors,vm_informed_max_abs_z,vm_informed_pct_vs_crossfit,vm_informed_driver_feature,vm_informed_driver_sensor
0,2024-08-21_09,9,9,17,4.27195,1.00000,late_minus_early__PlatenRFTuningCapacitor,PlatenRFTuningCapacitor
1,2024-07-19_01,5,1,17,3.88996,0.97674,std__PlatenRFTuningCapacitor,PlatenRFTuningCapacitor
2,2024-08-07_01,8,1,17,3.72676,0.96512,std__ForeLinePressure,ForeLinePressure
3,2024-08-07_08,8,8,17,3.41403,0.95349,late_minus_early__PlatenRFTuningCapacitor,PlatenRFTuningCapacitor
4,2024-07-02_02,1,2,17,3.29045,0.95349,robust_width__SourceRFReflectedPower,SourceRFReflectedPower
5,2024-07-11_05,4,5,17,3.67812,0.94186,std__PlatenRFReflectedPower,PlatenRFReflectedPower
6,2024-08-22_01,10,1,17,3.42692,0.91860,std__PlatenRFTuningCapacitor,PlatenRFTuningCapacitor
7,2024-07-11_03,4,3,17,3.37785,0.91860,std__PlatenRFReflectedPower,PlatenRFReflectedPower
8,2024-07-11_10,4,10,17,3.17297,0.90698,std__PlatenRFReflectedPower,PlatenRFReflectedPower
9,2024-08-01_10,6,10,17,3.07116,0.90698,robust_width__PlatenRFReflectedPower,PlatenRFReflectedPower


### 這個結果代表什麼？

09 grouped permutation importance 排名前 5 的 model-sensitive sensor groups 是：

1. `PlatenRFTuningCapacitor`
2. `MainActiveWindow`
3. `ForeLinePressure`
4. `SourceRFReflectedPower`
5. `PlatenRFReflectedPower`

它們可以對到 07 的 **17 個 engineered predictors**。這一節的 inner calibration 也通過 fixed outer feature universe 檢查，但它和 Pure OOD 的用途不同：這裡本來就使用了 09 的 supervised model importance，所以只能當 **VM-informed diagnostic**。

這次最高的幾個 standardized deviations 包括：

- `2024-08-21_09`（Lot 9）：max |z| = **4.27195**，cross-fitted percentile = **1.00000**，主要來自 `PlatenRFTuningCapacitor` 的 late-minus-early representation。
- `2024-07-19_01`（Lot 5）：max |z| = **3.88996**，percentile = **0.97674**。
- `2024-08-07_01`（Lot 8）：max |z| = **3.72676**，percentile = **0.96512**。
- `2024-08-07_08`（Lot 8）：max |z| = **3.41403**，percentile = **0.95349**。

這些數字只表示「wafer 在 VM 模型較依賴的 sensor groups 上偏離 reference distribution 的程度」。**這一節沒有建立 candidate flag**，也不會把它加進 Pure OOD 或 `any_process_review_evidence`。否則就會把 supervised model information 混進 process-only anomaly 定義。

## 6. Evidence Triangulation：把 process-only、Quality、VM error 與 VM-informed diagnostic 放在一起

現在才把不同來源的 evidence 合併。

- `residual_*`、`cross_lot_ood_*`、SPC：process-side evidence。
- Quality CV：measured outcome variation。
- Nested OOF error：held-out-Lot VM predictive error。
- `vm_informed_*`：target-informed diagnostic，只用來解釋模型脈絡。

Quality CV top-10% 和 |OOF error| top-10% 都只是 retrospective reference，不是 alarm rule。

Process-only continuous metrics 和 |OOF error| 的關係用 Spearman rho + Lot-cluster bootstrap 95% interval 描述。VM-informed score 另外列出，名稱上明確區分，避免讀者誤以為它是 independent process-only evidence。


In [6]:
tri = (
    process[["experiment_key", "lot_number", "wafer_number"]]
    .merge(
        residual_anomaly,
        on=["experiment_key", "lot_number", "wafer_number"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        cross_lot_ood,
        on=["experiment_key", "lot_number", "wafer_number"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        vm_informed,
        on=["experiment_key", "lot_number", "wafer_number"],
        how="left",
        validate="one_to_one",
    )
)

spc_cols = [
    "experiment_key",
    "n_flagged_features",
    "max_abs_standardized_residual",
    "n_mr_flagged_features",
    "max_mr_ratio_to_ucl",
    "has_mr_jump",
]
missing_spc_wafer_cols = [c for c in spc_cols if c not in spc_wafer.columns]
if missing_spc_wafer_cols:
    raise ValueError("spc_wafer 缺少欄位：" + ", ".join(missing_spc_wafer_cols))

tri = tri.merge(
    spc_wafer[spc_cols],
    on="experiment_key",
    how="left",
    validate="one_to_one",
)

for c in ["n_flagged_features", "n_mr_flagged_features"]:
    tri[c] = tri[c].fillna(0).astype(int)

if pd.api.types.is_bool_dtype(tri["has_mr_jump"]):
    tri["has_mr_jump"] = tri["has_mr_jump"].fillna(False).astype(bool)
else:
    # CSV merge 後可能變成 object；先用 nullable boolean 接住缺值，避免 pandas silent-downcast warning。
    tri["has_mr_jump"] = tri["has_mr_jump"].astype("boolean").fillna(False).astype(bool)

tri["has_spc_i_flag"] = tri["n_flagged_features"] > 0
tri["has_spc_mr_flag"] = tri["n_mr_flagged_features"] > 0
tri["has_duration_i_flag"] = tri["experiment_key"].isin(duration_i["experiment_key"])
tri["has_duration_mr_flag"] = tri["experiment_key"].isin(duration_mr["experiment_key"])

quality_cols = [
    "experiment_key", "mean_si_etch", "cv_si_etch_pct",
    "std_si_etch", "range_si_etch", "p90_width_si_etch",
]
tri = tri.merge(
    quality[quality_cols],
    on="experiment_key",
    how="left",
    validate="one_to_one",
)
tri = tri.merge(
    nested_oof[["experiment_key", "nested_process_pred", "nested_error", "abs_nested_error"]],
    on="experiment_key",
    how="left",
    validate="one_to_one",
)

cv90 = float(quality["cv_si_etch_pct"].quantile(0.90))
err90 = float(nested_oof["abs_nested_error"].quantile(0.90))
tri["high_quality_cv"] = tri["cv_si_etch_pct"] >= cv90
tri["high_nested_error"] = tri["abs_nested_error"] >= err90
tri["any_process_review_evidence"] = (
    tri["residual_review_candidate"]
    | tri["cross_lot_ood_candidate"]
    | tri["has_spc_i_flag"]
    | tri["has_spc_mr_flag"]
    | tri["has_duration_i_flag"]
    | tri["has_duration_mr_flag"]
)

# Association uses only wafers that have true quality labels / Nested OOF errors.
labeled = tri.loc[tri["abs_nested_error"].notna()].copy()

association_records = []
association_metrics = [
    "residual_mahalanobis",
    "residual_pca_recon_rmse",
    "ood_mahalanobis",
    "ood_pca_recon_rmse",
    "vm_informed_max_abs_z",
]

for metric_idx, metric in enumerate(association_metrics):
    rho, ci_low, ci_high, n_boot_valid = cluster_bootstrap_spearman_ci(
        labeled,
        x_col=metric,
        y_col="abs_nested_error",
        cluster_col="lot_number",
        n_boot=BOOTSTRAP_REPS,
        seed=BOOTSTRAP_SEED + metric_idx,
    )
    valid = labeled[[metric, "abs_nested_error", "lot_number"]].dropna()
    association_records.append({
        "metric": metric,
        "spearman_rho_with_abs_oof_error": rho,
        "abs_spearman_rho": abs(rho) if np.isfinite(rho) else np.nan,
        "lot_bootstrap_ci_low": ci_low,
        "lot_bootstrap_ci_high": ci_high,
        "bootstrap_valid_reps": int(n_boot_valid),
        "n_labeled_wafers": int(len(valid)),
        "n_lots": int(valid["lot_number"].nunique()),
    })

association = (
    pd.DataFrame(association_records)
    .sort_values("abs_spearman_rho", ascending=False)
    .reset_index(drop=True)
)

print("Quality CV retrospective 90th percentile：", round(cv90, 5), "%")
print("Nested absolute error retrospective 90th percentile：", round(err90, 5), "µm")
display(association.round(5))

strongest = association.iloc[0]
strong_rho = float(strongest["spearman_rho_with_abs_oof_error"])
strong_abs = abs(strong_rho)
if strong_abs < 0.20:
    association_strength = "很弱"
elif strong_abs < 0.40:
    association_strength = "偏弱到中等"
elif strong_abs < 0.60:
    association_strength = "中等"
else:
    association_strength = "偏強"
ci_crosses_zero = bool(strongest["lot_bootstrap_ci_low"] <= 0 <= strongest["lot_bootstrap_ci_high"])


process_only_assoc = association.loc[
    association["metric"] != "vm_informed_max_abs_z"
].copy()
strongest_process = process_only_assoc.iloc[0]
strongest_process_ci_crosses_zero = bool(
    strongest_process["lot_bootstrap_ci_low"]
    <= 0
    <= strongest_process["lot_bootstrap_ci_high"]
)

vm_assoc = association.loc[
    association["metric"] == "vm_informed_max_abs_z"
].iloc[0]

print(
    "Strongest process-only association：",
    strongest_process["metric"],
    f"rho={strongest_process['spearman_rho_with_abs_oof_error']:.5f}",
    f"95% lot-bootstrap interval=[{strongest_process['lot_bootstrap_ci_low']:.5f}, "
    f"{strongest_process['lot_bootstrap_ci_high']:.5f}]",
)
print(
    "Process-only strongest interval crosses 0：",
    strongest_process_ci_crosses_zero,
)
print(
    "VM-informed diagnostic association：",
    f"rho={vm_assoc['spearman_rho_with_abs_oof_error']:.5f}",
    f"95% lot-bootstrap interval=[{vm_assoc['lot_bootstrap_ci_low']:.5f}, "
    f"{vm_assoc['lot_bootstrap_ci_high']:.5f}]",
)

Quality CV retrospective 90th percentile： 8.44621 %
Nested absolute error retrospective 90th percentile： 0.2008 µm


,metric,spearman_rho_with_abs_oof_error,abs_spearman_rho,lot_bootstrap_ci_low,lot_bootstrap_ci_high,bootstrap_valid_reps,n_labeled_wafers,n_lots
0,ood_pca_recon_rmse,0.30948,0.30948,0.00848,0.60309,2000,88,10
1,ood_mahalanobis,0.23648,0.23648,-0.10785,0.59123,2000,88,10
2,residual_mahalanobis,0.11958,0.11958,-0.10355,0.32406,2000,88,10
3,residual_pca_recon_rmse,0.03013,0.03013,-0.18959,0.25073,2000,88,10
4,vm_informed_max_abs_z,0.01040,0.01040,-0.32242,0.34775,2000,88,10


Strongest process-only association： ood_pca_recon_rmse rho=0.30948 95% lot-bootstrap interval=[0.00848, 0.60309]
Process-only strongest interval crosses 0： False
VM-informed diagnostic association： rho=0.01040 95% lot-bootstrap interval=[-0.32242, 0.34775]


### 這個結果代表什麼？

這次 Quality CV 的 retrospective 90th percentile 是 **8.44621%**，Nested absolute OOF error 的 90th percentile 是 **0.20080 µm**。這兩個值只是拿來標記目前資料中相對高端的 observations，不是製程 specification。

在 88 片有 Quality / OOF 的 wafers 上，我把 process-side scores 和 `abs_nested_error` 做 Spearman association，再用 **Lot-cluster bootstrap 2,000 次**估 95% interval。實際結果是：

| Metric | Spearman rho with |OOF error| | 95% Lot-bootstrap interval |
|---|---:|---:|
| OOD PCA reconstruction RMSE | **0.30948** | **[0.00848, 0.60309]** |
| OOD Mahalanobis | 0.23648 | [-0.10785, 0.59123] |
| Residual Mahalanobis | 0.11958 | [-0.10355, 0.32406] |
| Residual PCA reconstruction RMSE | 0.03013 | [-0.18959, 0.25073] |
| VM-informed max |z| | 0.01040 | [-0.32242, 0.34775] |

Process-only 裡最明顯的是 **OOD PCA reconstruction RMSE，rho = 0.30948**。它的 bootstrap interval 沒有跨 0，但下界只有 **0.00848**，而且整個資料只有 10 Lots，所以我會把它解讀成「有一點正向關聯訊號，但強度不高，而且不夠穩定到可以說 model error 主要由 OOD 決定」。

其他 process-only metrics 的 interval 都跨 0，代表目前資料沒有支持穩定的單一 monotonic relationship。VM-informed max |z| 的 rho 更接近 0（**0.01040**），所以即使 wafer 在 model-sensitive sensors 上偏離，也不代表這片的 OOF error 一定比較大。

因此後面不應該把 anomaly / OOD score 當成 prediction-error score，也不應該把這些 correlation 當成因果證據。

## 7. 挑兩個案例看 evidence 為什麼不一定重疊

這裡不是建立新的 wafer ranking，而是用兩個案例幫助理解不同 evidence 的差異：

- **Case A｜Process + Model case**：已有 process-review evidence 的 labeled wafers 中，找 |Nested OOF error| 最大者。
- **Case B｜Quality contrast**：Quality CV 位於高端，但目前 process-review channels 沒有 flag 的 wafer。

注意：`vm_informed_*` 不納入 `any_process_review_evidence`，因為它是 supervised/model-informed diagnostic，不是獨立 process-only alarm channel。


In [7]:
case_a_pool = labeled.loc[labeled["any_process_review_evidence"]].copy()
if len(case_a_pool) == 0:
    case_a_pool = labeled.copy()
case_a = case_a_pool.sort_values("abs_nested_error", ascending=False).iloc[0].copy()

case_b_pool = labeled.loc[
    labeled["high_quality_cv"]
    & (~labeled["any_process_review_evidence"])
    & (labeled["experiment_key"] != case_a["experiment_key"])
].copy()

if len(case_b_pool) > 0:
    case_b = case_b_pool.sort_values("cv_si_etch_pct", ascending=False).iloc[0].copy()
    case_b_rule = "Quality CV 位於高端，而且目前 process-review channels 都沒有 flag"
    case_b_type = "B_Quality_Contrast"
    case_b_is_clean_contrast = True
else:
    case_b = (
        labeled.loc[labeled["experiment_key"] != case_a["experiment_key"]]
        .sort_values("cv_si_etch_pct", ascending=False)
        .iloc[0]
        .copy()
    )
    case_b_rule = "找不到完全無 process-review flag 的高 CV wafer；改選排除 Case A 後 Quality CV 最高者"
    case_b_type = "B_High_Quality_CV"
    case_b_is_clean_contrast = False

case_cols = [
    "experiment_key", "lot_number", "wafer_number",
    "residual_anomaly_consensus", "cross_lot_ood_consensus",
    "n_flagged_features", "n_mr_flagged_features",
    "cv_si_etch_pct", "abs_nested_error",
    "vm_informed_max_abs_z",
    "residual_driver_feature", "ood_driver_feature",
]

priority_cases = pd.DataFrame([
    case_a[case_cols],
    case_b[case_cols],
])
priority_cases.insert(0, "case_type", ["A_Process+Model", case_b_type])
priority_cases["selection_rule"] = [
    "有 process-review evidence 的 labeled wafers 中，|OOF error| 最大者",
    case_b_rule,
]

# Clean index because source row labels are not meaningful in the final artifact.
priority_cases = priority_cases.reset_index(drop=True)

display(priority_cases.round(5))


print("Case A：", case_a["experiment_key"], "|OOF error|=", round(float(case_a["abs_nested_error"]), 5))
print("Case B：", case_b["experiment_key"], "rule=", case_b_rule)
print(
    "解讀：Case A 是目前 process-review evidence 中 model error 最大的 labeled wafer；"
    "Case B 是否為乾淨 quality-only contrast，要依 cross-fitted OOD、SPC 和 residual flags 實際判定。"
)


,case_type,experiment_key,lot_number,wafer_number,residual_anomaly_consensus,cross_lot_ood_consensus,n_flagged_features,n_mr_flagged_features,cv_si_etch_pct,abs_nested_error,vm_informed_max_abs_z,residual_driver_feature,ood_driver_feature,selection_rule
0,A_Process+Model,2024-08-07_10,8,10,1,2,3,5,8.16625,0.57555,1.86279,Stat3_Etch_MV_Heater4Temp,late_minus_early__Heater2Temp,有 process-review evidence 的 labeled wafers 中，|...
1,B_Quality_Contrast,2024-08-01_07,6,7,0,0,0,0,8.94692,0.18857,2.60485,Stat3_Etch_MV_Pressure,robust_width__PlatenRFReflectedPower,Quality CV 位於高端，而且目前 process-review channels 都...


Case A： 2024-08-07_10 |OOF error|= 0.57555
Case B： 2024-08-01_07 rule= Quality CV 位於高端，而且目前 process-review channels 都沒有 flag
解讀：Case A 是目前 process-review evidence 中 model error 最大的 labeled wafer；Case B 是否為乾淨 quality-only contrast，要依 cross-fitted OOD、SPC 和 residual flags 實際判定。


### 這個結果代表什麼？

這次實際選到的兩個案例很適合說明「不同 evidence 不一定會一起出現」。

**Case A｜`2024-08-07_10`（Lot 8, wafer 10）**

- |Nested OOF error| = **0.57555 µm**，是有 process-review evidence 的 labeled wafers 中最大的一片。
- Residual consensus = **1 / 3**，所以它**不是** residual-review candidate。
- Cross-Lot OOD consensus = **2 / 3**，所以它是 Pure OOD candidate。
- SPC-I 有 **3 個 flagged features**，SPC-MR 有 **5 個 flagged features**。
- Quality CV = **8.16625%**，沒有超過本章的 90th-percentile CV reference。

所以這片比較像「Cross-Lot distribution extrapolation + SPC evidence + 很大的 VM error」同時出現，而不是所有 anomaly channels 都一致。這讓它很適合回頭做 process trace / equipment-event review，但仍不能單靠這些表格說是哪個 sensor 造成預測錯誤。

**Case B｜`2024-08-01_07`（Lot 6, wafer 7）**

- Quality CV = **8.94692%**，高於 90th-percentile reference 8.44621%。
- Residual consensus = **0**。
- Cross-Lot OOD consensus = **0**。
- SPC-I = **0**、SPC-MR = **0**。
- |Nested OOF error| = **0.18857 µm**，也沒有超過 0.20080 µm 的 90th-percentile error reference。

這是一個很乾淨的 contrast：**Quality spatial variation 可以偏高，但目前這些 process-review channels 不一定會同時 flag。** 所以 Quality anomaly、Process anomaly 和 Model error 必須分開看，不能互相替代。

## 8. 從 Lot 層級再看一次 evidence 是否一致

單片 wafer 看完後，我再把不同 evidence 聚合到 Lot：Residual candidates、Cross-Lot OOD、SPC I/MR、Quality CV、VM RMSE 和最大 prediction error。

這張表的用途是看 pattern，而不是把欄位加總成 ranking。不同指標的定義和尺度都不一樣，所以「某個 Lot 某一欄最高」只代表那個 evidence channel 比較突出，不等於這個 Lot 就最差。

In [8]:
lot_summary = (
    tri.groupby("lot_number", as_index=False)
    .agg(
        n_process_wafers=("experiment_key", "size"),
        residual_review_count=("residual_review_candidate", "sum"),
        mean_residual_mahalanobis=("residual_mahalanobis", "mean"),
        cross_lot_ood_count=("cross_lot_ood_candidate", "sum"),
        mean_ood_mahalanobis=("ood_mahalanobis", "mean"),
        spc_i_wafer_count=("has_spc_i_flag", "sum"),
        spc_mr_wafer_count=("has_spc_mr_flag", "sum"),
        n_quality_wafers=("mean_si_etch", "count"),
        mean_quality_cv=("cv_si_etch_pct", "mean"),
        high_quality_cv_count=("high_quality_cv", "sum"),
        vm_rmse=(
            "nested_error",
            lambda s: (
                float(np.sqrt(np.nanmean(np.asarray(s, dtype=float) ** 2)))
                if s.notna().any()
                else np.nan
            ),
        ),
        vm_max_abs_error=("abs_nested_error", "max"),
    )
    .sort_values("lot_number")
    .reset_index(drop=True)
)

display(lot_summary.round(5))

worst_vm = lot_summary.loc[lot_summary["vm_rmse"].idxmax()]
best_vm = lot_summary.loc[lot_summary["vm_rmse"].idxmin()]
max_ood_count = int(lot_summary["cross_lot_ood_count"].max())
most_ood_lots = lot_summary.loc[lot_summary["cross_lot_ood_count"] == max_ood_count, "lot_number"].astype(int).tolist()
most_ood_text = ", ".join(f"Lot {x}" for x in most_ood_lots)

if int(worst_vm["lot_number"]) in most_ood_lots:
    overlap_text = (
        f"VM RMSE 最高的 Lot {int(worst_vm['lot_number'])} 也在 OOD candidate 最多的 Lots 裡，"
        "代表兩種 evidence 在 Lot 層級有部分重疊，但仍不能把 OOD 當成 prediction error 的替代值。"
    )
else:
    overlap_text = (
        f"VM RMSE 最高的是 Lot {int(worst_vm['lot_number'])}，但 OOD candidates 最多的是 {most_ood_text}，"
        "所以『一個 Lot 很 OOD』和『VM 在這個 Lot 最難預測』不是同一件事。"
    )


print("Lot-level VM RMSE highest：", f"Lot {int(worst_vm['lot_number'])} = {worst_vm['vm_rmse']:.5f} µm")
print("Lot-level VM RMSE lowest：", f"Lot {int(best_vm['lot_number'])} = {best_vm['vm_rmse']:.5f} µm")
print("Cross-Lot OOD candidates most：", most_ood_text, f"({max_ood_count} wafers)")
print("解讀：", overlap_text)


,lot_number,n_process_wafers,residual_review_count,mean_residual_mahalanobis,cross_lot_ood_count,mean_ood_mahalanobis,spc_i_wafer_count,spc_mr_wafer_count,n_quality_wafers,mean_quality_cv,high_quality_cv_count,vm_rmse,vm_max_abs_error
0,1,10,0,4.20929,0,10.79555,1,2,9,8.07826,0,0.06516,0.11583
1,2,10,0,4.06710,0,9.67794,1,1,10,8.02445,1,0.10996,0.16586
2,3,10,0,4.22266,1,9.67501,2,4,10,8.22720,0,0.14304,0.22756
3,4,10,0,4.32545,0,9.46521,3,1,10,8.21564,1,0.11432,0.21846
4,5,10,0,4.13099,1,10.53817,3,3,10,8.14189,1,0.13771,0.27007
5,6,10,1,4.20019,0,10.09286,1,1,10,8.31099,1,0.11752,0.27963
6,7,6,0,3.67758,0,10.01138,0,1,6,8.25337,0,0.11837,0.20857
7,8,10,2,4.94138,3,12.26297,6,4,10,8.30709,2,0.22384,0.57555
8,9,10,0,3.52332,0,9.54259,0,3,9,8.31790,2,0.06980,0.12378
9,10,10,0,3.96038,0,9.40992,2,3,4,8.31193,1,0.08315,0.10814


Lot-level VM RMSE highest： Lot 8 = 0.22384 µm
Lot-level VM RMSE lowest： Lot 1 = 0.06516 µm
Cross-Lot OOD candidates most： Lot 8 (3 wafers)
解讀： VM RMSE 最高的 Lot 8 也在 OOD candidate 最多的 Lots 裡，代表兩種 evidence 在 Lot 層級有部分重疊，但仍不能把 OOD 當成 prediction error 的替代值。


### 這個結果代表什麼？

把 evidence 聚合到 Lot 後，**Lot 8 是目前多個 process / model-risk signals 最集中的 Lot**：

- 10 片 Process wafers 中有 **2 片 residual-review candidates**。
- 有 **3 片 Cross-Lot OOD candidates**，是所有 Lots 中最多。
- SPC-I 有 **6 片 wafers**出現 flag，SPC-MR 有 **4 片**。
- 有 **2 片** Quality CV 位於全體 labeled wafers 的高端。
- VM RMSE = **0.22384 µm**，是 10 Lots 中最高。
- 最大 |OOF error| = **0.57555 µm**，也來自 Lot 8。

相對地，Lot 1 的 VM RMSE 最低，只有 **0.06516 µm**。另外 Lot 9 的 mean Quality CV 其實略高（約 **8.31790%**），但它沒有 residual / OOD candidates，而且 VM RMSE 只有 **0.06980 µm**。

所以 Lot 8 確實值得列為後續 investigation priority，因為不同 evidence 在這個 Lot 有明顯重疊；但這仍然只是「值得優先查」而不是「Lot 8 已被證明發生某個特定製程故障」。尤其 OOD、SPC、Quality 和 VM error 的定義不同，不能把它們直接加總成一個 anomaly severity score。

## 9. 儲存正式 artifacts，讓 11 只負責統整

最後把每種 evidence 分開存檔。最重要的 schema 原則是：

- `10_cross_lot_ood_scores.csv`：**只含 Pure Process OOD**，不放 09-derived model-sensitive 欄位。
- `10_vm_informed_diagnostic.csv`：獨立保存 target-informed diagnostic。
- `10_anomaly_triangulation.csv`：可以把兩類 evidence 合併供 11 比較，但欄位名稱仍保持來源差異。
- methodology JSON 明確記錄 Pure OOD 不使用 Quality target / 08 error / 09 importance。

每個檔案寫出後都重新讀回檢查 rows 和 schema。


In [9]:
model_risk_overlap = labeled[[
    "experiment_key", "lot_number", "wafer_number",
    "residual_review_candidate", "cross_lot_ood_candidate",
    "has_spc_i_flag", "has_spc_mr_flag",
    "high_quality_cv", "high_nested_error",
    "cv_si_etch_pct", "abs_nested_error",
    "vm_informed_max_abs_z", "vm_informed_pct_vs_crossfit",
]].copy()

csv_outputs = {
    PROCESSED_DATA_DIR / "10_residual_anomaly_scores.csv": residual_anomaly,
    PROCESSED_DATA_DIR / "10_cross_lot_ood_scores.csv": cross_lot_ood,
    PROCESSED_DATA_DIR / "10_ood_fold_thresholds.csv": ood_fold_thresholds,
    PROCESSED_DATA_DIR / "10_vm_informed_diagnostic.csv": vm_informed,
    PROCESSED_DATA_DIR / "10_anomaly_triangulation.csv": tri,
    PROCESSED_DATA_DIR / "10_anomaly_by_lot.csv": lot_summary,
    PROCESSED_DATA_DIR / "10_process_vs_model_error_association.csv": association,
    PROCESSED_DATA_DIR / "10_priority_cases.csv": priority_cases,
    PROCESSED_DATA_DIR / "10_model_risk_overlap.csv": model_risk_overlap,
}

methodology = {
    "process_wafers": int(n_process),
    "quality_labeled_wafers": int(n_quality),
    "lots": int(n_lots),
    "residual_feature_rule": (
        "只使用 valid_lots == Lot 總數且 n_valid_wafers == Process wafer 總數的 features；valid_lot=False placeholder residual 不納入。"
    ),
    "residual_feature_count": int(len(residual_features)),
    "residual_excluded_features": excluded_residual_features["feature"].astype(str).tolist(),
    "residual_pca_components_90pct": int(k_r),
    "residual_pca_matches_notebook_06": True,
    "residual_candidate_rule": (
        "三個 residual metrics 使用完整 Process population 的 retrospective 95th percentile；至少 2/3 flags 才列為 review candidate。"
    ),
    "pure_cross_lot_ood_feature_source": "Notebook 07 process engineered features only",
    "pure_cross_lot_ood_uses_quality_target": False,
    "pure_cross_lot_ood_uses_notebook08_error": False,
    "pure_cross_lot_ood_uses_notebook09_importance": False,
    "pure_cross_lot_ood_fixed_outer_feature_universe": True,
    "cross_lot_ood_fit_rule": (
        "每次完整 held-out 一個 Lot；outer-training Lots 先決定固定 near-constant feature universe。"
        "Inner calibration 與 final outer detector 都使用同一套 outer-selected predictors；"
        "各自的 scaler、Ledoit-Wolf covariance、PCA 與 PCA component count 只由各自 training Lots fit。"
    ),
    "cross_lot_ood_threshold_rule": (
        "Outer reference Lots 內再做 Inner Leave-One-Lot-Out；所有 inner folds 固定使用 outer-training-only feature universe，"
        "reference wafer 由未看過其 Lot 的 detector 產生 cross-fitted score，再以這些 scores 建 95th-percentile thresholds；"
        "至少 2/3 flags 才列為 candidate。"
    ),
    "cross_lot_ood_calibration_limitation": (
        "group-aware cross-fitted empirical calibration；不是 exact conformal calibration、獨立 production control limit 或正式 false-positive-rate guarantee。"
    ),
    "vm_informed_diagnostic_rule": (
        "09 global supervised grouped permutation importance 選 top-5 sensor groups；每個 outer fold 先由 outer-training Lots 固定 model-sensitive engineered-feature universe，"
        "inner LOGO reference 與 held-out scoring 使用同一套 outer-selected features，各自的 scaling 只由對應 training Lots fit。"
        "此 channel 明確屬 target-informed diagnostic，不建立 anomaly candidate flag。"
    ),
    "vm_informed_top_sensors": top_model_sensors,
    "association_rule": (
        "labeled wafers 上計算 Spearman rho，並用 Lot-cluster bootstrap 估 95% interval；process-only metrics 與 VM-informed diagnostic 在命名上分開。"
    ),
    "bootstrap_reps": int(BOOTSTRAP_REPS),
    "bootstrap_seed": int(BOOTSTRAP_SEED),
    "residual_review_candidates": int(residual_anomaly["residual_review_candidate"].sum()),
    "cross_lot_ood_candidates": int(cross_lot_ood["cross_lot_ood_candidate"].sum()),
}

if not ood_fold_thresholds["calibration_uses_fixed_outer_feature_universe"].all():
    raise ValueError(
        "10_ood_fold_thresholds 中存在未使用 fixed outer feature universe 的 fold"
    )

validation = []
for path, df in csv_outputs.items():
    df.to_csv(path, index=False)
    chk = pd.read_csv(path)
    if len(chk) != len(df) or list(chk.columns) != list(df.columns):
        raise ValueError(f"10 output validation failed: {path.name}")
    validation.append({"artifact": path.name, "rows": len(chk), "schema_match": True})

methodology_path = PROCESSED_DATA_DIR / "10_anomaly_methodology.json"
with open(methodology_path, "w", encoding="utf-8") as f:
    json.dump(methodology, f, ensure_ascii=False, indent=2)
with open(methodology_path, "r", encoding="utf-8") as f:
    methodology_check = json.load(f)
if methodology_check != methodology:
    raise ValueError("10_anomaly_methodology.json read-back validation failed")

validation.append({"artifact": methodology_path.name, "rows": 1, "schema_match": True})
validation_df = pd.DataFrame(validation)
display(validation_df)

,artifact,rows,schema_match
0,10_residual_anomaly_scores.csv,96,True
1,10_cross_lot_ood_scores.csv,96,True
2,10_ood_fold_thresholds.csv,10,True
3,10_vm_informed_diagnostic.csv,96,True
4,10_anomaly_triangulation.csv,96,True
5,10_anomaly_by_lot.csv,10,True
6,10_process_vs_model_error_association.csv,5,True
7,10_priority_cases.csv,2,True
8,10_model_risk_overlap.csv,88,True
9,10_anomaly_methodology.json,1,True


### 這個結果代表什麼？

最後一格已經把 Notebook 10 的正式結果重新寫出並讀回驗證。**9 個 CSV + 1 個 methodology JSON 全部 `schema_match = True`**，row count 也和本章實際分析一致：

| Artifact | Rows |
|---|---:|
| `10_residual_anomaly_scores.csv` | 96 |
| `10_cross_lot_ood_scores.csv` | 96 |
| `10_ood_fold_thresholds.csv` | 10 |
| `10_vm_informed_diagnostic.csv` | 96 |
| `10_anomaly_triangulation.csv` | 96 |
| `10_anomaly_by_lot.csv` | 10 |
| `10_process_vs_model_error_association.csv` | 5 |
| `10_priority_cases.csv` | 2 |
| `10_model_risk_overlap.csv` | 88 |
| `10_anomaly_methodology.json` | 1 |

其中責任分工也有保持乾淨：Pure OOD scores 沒有放 09-derived model-sensitive 欄位；VM-informed diagnostic 另外存；association、priority cases 和 lot summary 則是後面的 evidence integration artifacts。

因此 Notebook 11 可以直接讀這些正式輸出做 final findings，不需要再自己重算 anomaly / OOD，也不應該重新發明另一套 candidate rule。

## 10. 整體結論

這次用實際 `processed` artifacts 完整重跑後，Notebook 10 的結果可以整理成下面幾點。

### 1. Residual anomaly 和 Cross-Lot OOD 確實不是同一件事

96 片 Process wafers 中，Residual Review Candidates 有 **3 片**，Pure Cross-Lot OOD Candidates 有 **5 片**，兩邊只有 `2024-08-07_08` 和 `2024-08-07_09` **2 片重疊**。

所以 Lot 內扣掉 sequence trend 後的 unusualness，和相對其他 Lots 的 distribution extrapolation，應該繼續分成兩個 evidence channels。

### 2. Lot 8 是目前最值得優先回查的 Lot

Lot 8 同時有 **2 個 residual candidates、3 個 OOD candidates、6 個 SPC-I wafers、4 個 SPC-MR wafers**，而且它的 VM RMSE **0.22384 µm** 是所有 Lots 中最高，最大 OOF error 也達 **0.57555 µm**。

這代表多種 evidence 在 Lot 8 有實際重疊，因此後續若要回頭看 raw process trace、equipment event、maintenance record 或 recipe context，Lot 8 是合理的優先順序。這裡的「優先」是 investigation priority，不是 defect diagnosis。

### 3. Process unusualness 不能直接解釋 Virtual Metrology error

在 88 片 labeled wafers 上，最強的 process-only association 是 OOD PCA reconstruction RMSE 和 |OOF error|，Spearman rho = **0.30948**，Lot-bootstrap 95% interval = **[0.00848, 0.60309]**。其他 process-only metrics 的 interval 都跨 0。

因此目前最多只能說某些 cross-lot extrapolation 和 model error 有部分關聯訊號，不能說 anomaly 越高就一定預測越差，也不能把 OOD score 當作 VM error 的 proxy。

### 4. 兩個 case 說明 Quality、Process、Model 三層 evidence 必須分開

`2024-08-07_10` 有 OOD + SPC evidence，同時也是本章 Case A 中 |OOF error| 最大的一片；但它不是 residual-review candidate。反過來，`2024-08-01_07` 的 Quality CV 很高，卻沒有 residual、OOD 或 SPC flags，OOF error 也沒有進入最高 10%。

這表示「Quality 特別差」、「Process pattern 特別少見」和「Model 特別難預測」在這份資料裡不是同一個事件。

### 5. 本章最後保留的限制

- 資料只有 **96 Process wafers / 10 Lots**，cross-lot reference 和 bootstrap interval 都仍受小樣本限制。
- Residual screening 使用 05 的 Phase-I retrospective sequence adjustment，不是 production-time prospective detector。
- OOD threshold 是 group-aware cross-fitted empirical reference，不是 exact conformal guarantee，也不是正式 manufacturing alarm limit。
- 09-derived VM-informed diagnostic 是 supervised / target-informed context，所以沒有混進 Pure OOD candidate rule。
- Feature driver 只能指出「哪個 representation 在這個 statistical score 裡偏得最多」，不能直接當 causal root cause。

**Notebook 10 到這裡的任務已經完成：把 Residual Anomaly、Pure Cross-Lot OOD、SPC、Quality、VM error 和 VM-informed context 分開定義，再在 wafer / Lot 層級做 evidence triangulation。Notebook 11 接下來只需要統整這些已驗證的 artifacts，形成整個專案的 final findings。**